<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/main/01_MAPAS_TFG2_2_R00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install, autenticate, Initialize**

In [ ]:
#!pip install geemap

In [ ]:
#import libraries
import ee
import geemap
import geopandas as gpd
import os
import time
import pandas as pd

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#autenticate and initialize
ee.Authenticate()
ee.Initialize(project='thermal-luizadraeger')

**Import - map base and set center**

In [ ]:
#Create map
Map = geemap.Map(basemap="Esri.WorldImagery") #Adding basemaps - Map = geemap.Map(basemap="Esri.WorldImagery")


**Define region of interest**

In [ ]:
#region_of_interest - Poligon
'''
# Define the region of interest (ROI) for the river - poligon
ROI = ee.Geometry.Polygon([
    [
        [-42.97880, -22.42410],  # canto noroeste
        [-42.97315, -22.42410],  # canto nordeste
        [-42.97315, -22.43940],  # canto sudeste
        [-42.97880, -22.43940],  # canto sudoeste
        [-42.97880, -22.42410]   # fechamento do polígono
    ]
])

# Add the ROI to the map for visualization
Map.centerObject(ROI, 11)  # Centraliza o mapa na ROI com zoom 10
Map.addLayer(ROI, {'color': 'blue'}, 'Region')
'''

# Define the region of interest (ROI) for Teresopolis - poligon
ROI = ee.Geometry.Polygon([
  [
    [-43.3200, -22.7450],
    [-43.3200, -22.7290],
    [-43.3000, -22.7290],
    [-43.3000, -22.7450],
    [-43.3200, -22.7450]
  ]
])


# Add the ROI to the map for visualization
Map.centerObject(ROI, 11)  # Centraliza o mapa na ROI com zoom 10
Map.addLayer(ROI, {'color': 'red'}, 'Region of Interest')


**Mapa Base**

In [ ]:
# Load the basemap as an ee.Image
Map = geemap.Map(basemap="Esri.WorldImagery") #Adding basemaps - Map = geemap.Map(basemap="Esri.WorldImagery")

# Now you can clip it to your ROI
Mapclip = basemap.clip(ROI)

**Hidrografia**

In [ ]:
#add rivers from Teresópolis County
# Definindo os diretórios dos shapefiles
shapefile_dirs = {
    "Rio Paquequer": '/content/drive/MyDrive/03_TFG/00_TFG_RESILIÊNCIA/00_MAPAS QGIS/00_SHAPEFILES/Hidrografia - Rio Paquequer',
}

#G:\Meu Drive\03_TFG\00_TFG_RESILIÊNCIA\00_MAPAS QGIS\00_SHAPEFILES\Hidrografia - Rio Paquequer

print("Verificando componentes dos shapefiles:")
gdf_dict = {}

for layer_name, dir_path in shapefile_dirs.items():
    try:
        # Procurar arquivo .shp no diretório
        shp_file = next((f for f in os.listdir(dir_path) if f.endswith('.shp')), None)
        if not shp_file:
            print(f"\nCamada: {layer_name}\n  Arquivo .shp não encontrado na pasta.")
            continue

        base_name = os.path.splitext(shp_file)[0]
        required_extensions = ['shp', 'shx', 'dbf', 'prj']
        required_files = [f"{base_name}.{ext}" for ext in required_extensions]

        print(f"\nCamada: {layer_name}")
        missing_files = [f for f in required_files if not os.path.exists(os.path.join(dir_path, f))]

        if missing_files:
            print(f"  Arquivos faltando: {missing_files}")
            continue

        # Carregar shapefile
        full_path = os.path.join(dir_path, shp_file)
        gdf = gpd.read_file(full_path)

        # Reprojetar para WGS84 se necessário
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')
            print(f"  {layer_name} reprojetado para WGS84")

        gdf_dict[layer_name] = gdf
        print(f"  {layer_name} carregado com sucesso! (CRS: {gdf.crs}, Feições: {len(gdf)})")

    except Exception as e:
        print(f"  Erro ao carregar {layer_name}: {str(e)}")

# Criar mapa (apenas se pelo menos uma camada foi carregada)
if gdf_dict:
    Map = geemap.Map()

    # Estilos para cada camada
    styles = {
        "Rio Paquequer": {'color': 'blue', 'width': 2},
    }

    # Adicionar cada camada ao mapa
    for layer_name, gdf in gdf_dict.items():
        geojson = gdf.__geo_interface__
        ee_feature_collection = geemap.geojson_to_ee(geojson)
        Map.addLayer(ee_feature_collection, styles[layer_name], layer_name)

    # Ajustar visualização
    combined_gdf = gpd.GeoDataFrame(pd.concat(gdf_dict.values(), ignore_index=True))
    Map.centerObject(geemap.geojson_to_ee(combined_gdf.__geo_interface__), zoom=10)

    Map.addLayerControl()
else:
    print("\nNenhuma camada válida foi carregada. Verifique os arquivos de origem.")


Verificando componentes dos shapefiles:

Camada: Rio Paquequer
  Rio Paquequer reprojetado para WGS84
  Rio Paquequer carregado com sucesso! (CRS: EPSG:4326, Feições: 61)


**Digital Elevation Model (DEM) Slope and Aspect**

In [ ]:
#DEM
dem = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(ROI)
dem_vis = {
    'min': 16,
    'max': 2162,
}
Map.addLayer(dem, dem_vis, 'DEM')


#Slope
slope = ee.Terrain.slope(dem).clip(ROI)
slope_vis = {
    'min': 0,
    'max': 90,
    'palette': [
        '#440154',
        '#46327e',
        '#365c8d',
        '#277f8e',
        '#1fa187',
        '#4ac16d',
        '#a0da39',
        '#fde725'
    ]
}
Map.addLayer(slope, slope_vis, 'Slope')

#Aspect
aspect = ee.Terrain.aspect(dem).clip(ROI)
aspect_vis = {
    'min': 0,
    'max': 360,
    'palette': [
        '#0000ff',
        '#00ff00',
        '#ffff00',
        '#ff7f00',
        '#ff0000'
    ]
}
Map.addLayer(aspect, aspect_vis, 'Aspect')

**Countour Lines**

In [ ]:
# Definir o intervalo e a sequência de linhas de contorno
lines = ee.List.sequence(0, 5000, 10)

# Função para gerar as linhas de contorno
def generate_contour(line):
    dem_contour = dem \
        .convolve(ee.Kernel.gaussian(5, 3)) \
        .subtract(ee.Image.constant(line)) \
        .zeroCrossing() \
        .multiply(ee.Image.constant(line)) \
        .toFloat()

    return dem_contour.mask(dem_contour)

# Aplicar a função em cada linha da sequência
contourlines = lines.map(generate_contour)

# Criar a coleção de imagens e gerar o mosaico
contour_line = ee.ImageCollection.fromImages(contourlines).mosaic().clip(ROI)
Map.add_ee_layer(contour_line, {'palette': 'red'}, 'Contour Lines')

**Building Footprint**

In [ ]:
t = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')

t_065_070 = t.filter('confidence >= 0.65 and confidence <= 0.70').filterBounds(ROI);
t_070_075 = t.filter('confidence >= 0.70 and confidence <= 0.75').filterBounds(ROI);
t_075_075 = t.filter('confidence >= 0.75').filterBounds(ROI);


#Map.addLayer(t_065_070, {'color': 'FF0000'}, 'Buildings confidence [0.65; 0.70]');
#Map.addLayer(t_070_075, {'color': '00FF00'}, 'Buildings confidence [0.70; 0.75]');
#Map.addLayer(t_075_075, {'color': '0000FF'}, 'Buildings confidence >= 0.75');

# Instead of using a set, use FeatureCollection.merge to combine the three collections
buildings = t_065_070.merge(t_070_075).merge(t_075_075)
Map.addLayer(buildings, {'color': 'blue'}, 'Buildings')

**Map visualization**

In [ ]:
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

**Exportar Informações**

In [ ]:
# Exportar para o Google Drive
task = ee.batch.Export.table.toDrive(
    collection=buildings,
    description='export_buildings',
    folder='Solano-bienal',        # Nome da pasta no seu Google Drive
    fileNamePrefix='buildings_roi',
    fileFormat='GeoJSON'             # Pode ser 'CSV', 'SHP', 'GeoJSON', etc.
)

task.start()

# Checar status da tarefa até finalizar
while task.active():
    print('Status:', task.status()['state'])
    time.sleep(10)

print('Exportação finalizada com status:', task.status())


Status: READY
Status: RUNNING
Status: COMPLETED
Exportação finalizada com status: {'state': 'COMPLETED', 'description': 'export_buildings', 'priority': 100, 'creation_timestamp_ms': 1754577215316, 'update_timestamp_ms': 1754577236411, 'start_timestamp_ms': 1754577222335, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1Eyk62Y8VqMN3iNRxOKJGE3DU4cvtBlus'], 'attempt': 1, 'batch_eecu_usage_seconds': 15.510440826416016, 'id': 'D7VR7JJGI3SU43YZKX2FKOU7', 'name': 'projects/thermal-luizadraeger/operations/D7VR7JJGI3SU43YZKX2FKOU7'}


In [ ]:
# Converter a ROI em FeatureCollection
roi_fc = ee.FeatureCollection([ee.Feature(ROI)])

# Exportar para o Google Drive
task = ee.batch.Export.table.toDrive(
    collection=roi_fc,
    description='RegionOfInterest',
    folder='Solano-bienal',
    fileNamePrefix='ROI',
    fileFormat='GeoJSON'
)

task.start()

# Checar status da tarefa até finalizar
while task.active():
    print('Status:', task.status()['state'])
    time.sleep(10)

print('Exportação finalizada com status:', task.status())


Status: READY
Exportação finalizada com status: {'state': 'COMPLETED', 'description': 'RegionOfInterest', 'priority': 100, 'creation_timestamp_ms': 1754577247070, 'update_timestamp_ms': 1754577254718, 'start_timestamp_ms': 1754577252236, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1Eyk62Y8VqMN3iNRxOKJGE3DU4cvtBlus'], 'attempt': 1, 'batch_eecu_usage_seconds': 0.000302431988529861, 'id': 'SROIALOL35CF3L3YH3O2BV3V', 'name': 'projects/thermal-luizadraeger/operations/SROIALOL35CF3L3YH3O2BV3V'}


In [ ]:
# Converter a imagem de contorno em feições vetoriais
contour_features = contour_line.reduceToVectors(
    reducer=ee.Reducer.countEvery(),
    maxPixels=1e8
)

# Exportar para o Google Drive
task = ee.batch.Export.table.toDrive(
    collection=contour_features,
    description='contour_lines',
    folder='Solano-bienal',
    fileNamePrefix='contour_lines',
    fileFormat='SHP'
)

task.start()

# Checar status da tarefa até finalizar
while task.active():
    print('Status:', task.status()['state'])
    time.sleep(10)  # Espera 10 segundos antes de checar de novo

print('Exportação finalizada com status:', task.status())

Status: READY
Status: READY
Exportação finalizada com status: {'state': 'FAILED', 'description': 'contour_lines', 'priority': 100, 'creation_timestamp_ms': 1754577257791, 'update_timestamp_ms': 1754577274188, 'start_timestamp_ms': 1754577272408, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.446923851, 'error_message': "Image.reduceToVectors: First band ('elevation') of image must be integral.", 'id': 'I6RLSZ3BEVINH7D7OHFO42M5', 'name': 'projects/thermal-luizadraeger/operations/I6RLSZ3BEVINH7D7OHFO42M5'}


In [ ]:
# Exportar para o Google Drive
task = ee.batch.Export.image.toDrive(
    image=dem,
    description='Exportar_MDE',
    folder='Solano-bienal',
    fileNamePrefix='Modelo_de_elevacao_digital',

    region=ROI.bounds().getInfo()['coordinates'],
    scale=30,                 # resolução (30m para SRTM)
    crs='EPSG:4326',          # sistema de coordenadas
    maxPixels=1e13            # aumenta se necessário
)

task.start()

# Checar status da tarefa até finalizar
while task.active():
    print('Status:', task.status()['state'])
    time.sleep(10)

print('Exportação finalizada com status:', task.status())

Status: READY
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Exportação finalizada com status: {'state': 'COMPLETED', 'description': 'Exportar_MDE', 'priority': 100, 'creation_timestamp_ms': 1754577279061, 'update_timestamp_ms': 1754577373183, 'start_timestamp_ms': 1754577287950, 'task_type': 'EXPORT_IMAGE', 'destination_uris': ['https://drive.google.com/#folders/1Eyk62Y8VqMN3iNRxOKJGE3DU4cvtBlus'], 'attempt': 1, 'batch_eecu_usage_seconds': 0.039961546659469604, 'id': 'FBW6CWGM77U5P7RCRCZNLRUT', 'name': 'projects/thermal-luizadraeger/operations/FBW6CWGM77U5P7RCRCZNLRUT'}
